# Content Creator Sidekick

A LangGraph-powered agent that researches, writes, and delivers content for any platform:
Twitter/X, LinkedIn, Instagram, YouTube scripts, podcast scripts, blogs, and newsletters.

**Features:**
- 12 tools (search, browser, YouTube, files, Python REPL, hashtags, word counter, etc.)
- Tone picker: hype, casual, professional, educational, storytelling
- Video script support: YouTube long-form, Shorts/TikTok/Reels, podcast
- Evaluator loop: checks quality, tone, platform fit, and success criteria
- Saves all output to `sandbox/content/`

This notebook walks through every piece step by step so you can understand how it connects.

## Step 0: Setup and Imports

In [ ]:
from typing import Annotated, List, Any, Optional, Dict
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from IPython.display import Image, display
from dotenv import load_dotenv
from datetime import datetime
import gradio as gr
import nest_asyncio
import uuid

nest_asyncio.apply()
load_dotenv(override=True)
print("Imports loaded.")

## Step 1: Load All 12 Tools

These come from `sidekick_tools.py`. Two functions:
- `playwright_tools()` — starts a visible Chromium browser and returns browser tools + handles
- `other_tools()` — returns the other 10 tools (search, YouTube, files, REPL, hashtags, etc.)

After this cell, `tools` is a single list the worker LLM and ToolNode both use.

In [ ]:
from sidekick_tools import playwright_tools, other_tools

tools_list, browser, playwright = await playwright_tools()
tools_list += await other_tools()

print(f"Loaded {len(tools_list)} tools:")
for t in tools_list:
    print(f"  - {t.name}: {t.description[:80]}")

## Step 2: Test a Few Tools

Let's make sure the tools work before wiring them into the graph.

In [ ]:
# Test the word counter
from sidekick_tools import count_words_and_chars

sample = "AI agents are the future of software. Every developer should learn LangGraph."
print(count_words_and_chars(sample))

In [ ]:
# Test the hashtag generator
from sidekick_tools import generate_hashtags

print(generate_hashtags("AI agents, LangGraph, Python, automation"))

In [ ]:
# Test the datetime tool
from sidekick_tools import get_datetime_info

print(get_datetime_info("now"))

## Step 3: Define State

The shared notepad every node reads and updates.

- `messages` — chat history (uses `add_messages` reducer to merge correctly)
- `success_criteria` — what the user wants (typed in Gradio)
- `tone` — how the content should sound (hype, casual, professional, educational, storytelling)
- `feedback_on_work` — evaluator's notes if the draft was rejected
- `success_criteria_met` / `user_input_needed` — routing flags

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    tone: Optional[str]
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    user_input_needed: bool

print("State defined.")

## Step 4: Tone System + Content Formats

The worker uses these to know HOW to write and WHAT structure each platform needs.

In [ ]:
TONE_OPTIONS = {
    "hype": "High energy, motivational, punchy. Use power words, short sentences, exclamation marks. Make the reader feel fired up.",
    "casual": "Casual and conversational, like talking to a friend. Use contractions, simple language, maybe some humor.",
    "professional": "Professional but approachable. Clean, polished, confident. Good for LinkedIn and business content.",
    "educational": "Clear, structured, teacher-style. Break things down step by step. Use examples and analogies.",
    "storytelling": "Narrative and emotional. Draw the reader in with a story arc — hook, tension, resolution. Personal and relatable.",
}

DEFAULT_TONE = "hype"

CONTENT_FORMATS = """
You know the following content formats and their rules:

SOCIAL MEDIA:
- Twitter/X: Max 280 characters. Punchy, one key idea. Hashtags at the end.
- LinkedIn: Up to ~3,000 characters. Professional hook in first 2 lines (before 'see more'). Use line breaks.
- Instagram caption: Up to 2,200 characters. Visual storytelling. Hashtags (up to 30) at end.
- TikTok caption: Up to 2,200 characters. Short, catchy, trend-aware.

VIDEO SCRIPTS:
- YouTube long-form (5-15 min): HOOK (first 30s) → INTRO → MAIN CONTENT (3-5 sections) → CTA → OUTRO. Include timestamps.
- YouTube Shorts / TikTok / Reels (under 60s): HOOK (first 3s) → ONE key point → PAYOFF. Fast-paced.

PODCAST SCRIPTS:
- COLD OPEN → INTRO → SEGMENTS (2-4 points) → LISTENER CTA → OUTRO. Natural speaking voice. Include [PAUSE] and [EMPHASIS] cues.

BLOG / NEWSLETTER:
- Headline → Hook paragraph → Subheaded sections → Conclusion with CTA. Short paragraphs. Bold key phrases.

ALWAYS save finished content to sandbox/content/ with a descriptive filename.
"""

print("Tone options and content formats defined.")
print(f"Available tones: {list(TONE_OPTIONS.keys())}")

## Step 5: Evaluator Output Schema

Forces the evaluator LLM to return structured data, not free text.

In [ ]:
class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on content quality, tone, structure, and whether it meets the success criteria")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(description="True if more input is needed from the user, or clarifications, or the assistant is stuck")

print("EvaluatorOutput schema defined.")

## Step 6: Set Up the Two LLMs

- **Worker LLM**: bound to all 12 tools so it can emit tool_calls
- **Evaluator LLM**: bound to `EvaluatorOutput` so it returns structured judgment

In [ ]:
worker_llm = ChatOpenAI(model="gpt-4o-mini")
worker_llm_with_tools = worker_llm.bind_tools(tools_list)

evaluator_llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorOutput)

print("Worker LLM: gpt-4o-mini with", len(tools_list), "tools bound")
print("Evaluator LLM: gpt-4o-mini with structured output")

## Step 7: Worker Node

The do-er. It:
1. Builds a system prompt with the chosen tone + content format rules + success criteria
2. If previous feedback exists, appends "try again" instructions
3. Calls the LLM (which may return tool calls or a text reply)

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    tone_key = (state.get("tone") or DEFAULT_TONE).lower().strip()
    tone_description = TONE_OPTIONS.get(tone_key, TONE_OPTIONS[DEFAULT_TONE])

    system_message = f"""You are a Content Creator Sidekick — an expert at writing viral, engaging content for any platform.

YOUR TONE: {tone_key}
{tone_description}
Every piece of content MUST match this tone. If unsure of tone, ASK the user. Available tones: {', '.join(TONE_OPTIONS.keys())}.

CONTENT FORMATS:
{CONTENT_FORMATS}

WORKFLOW: Research first → Draft → Check word/char limits → Save to sandbox/content/ → Notify user.

Current date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

SUCCESS CRITERIA:
{state['success_criteria']}

If you have a question, clearly state it. Otherwise reply with the final content."""

    if state.get("feedback_on_work"):
        system_message += f"\n\nPREVIOUS ATTEMPT REJECTED:\n{state['feedback_on_work']}\nPlease fix and improve."

    found_system = False
    messages = state["messages"]
    for msg in messages:
        if isinstance(msg, SystemMessage):
            msg.content = system_message
            found_system = True

    if not found_system:
        messages = [SystemMessage(content=system_message)] + messages

    response = worker_llm_with_tools.invoke(messages)
    return {"messages": [response]}

print("Worker node defined.")

## Step 8: Worker Router

After the worker runs, check: did it ask for tools or give a text reply?
- Tool calls → go to `tools` node
- Text reply → go to `evaluator`

In [ ]:
def worker_router(state: State) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "evaluator"

print("Worker router defined.")

## Step 9: Evaluator Node

The judge. It reads the full conversation + success criteria + requested tone, then decides:
- Criteria met? → stop
- User input needed? → stop
- Neither? → send worker back with feedback

In [ ]:
def format_conversation(messages: List[Any]) -> str:
    conversation = "Conversation history:\n\n"
    for msg in messages:
        if isinstance(msg, HumanMessage):
            conversation += f"User: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            text = msg.content or "[Tools use]"
            conversation += f"Assistant: {text}\n"
    return conversation


def evaluator(state: State) -> State:
    last_response = state["messages"][-1].content
    tone_key = (state.get("tone") or DEFAULT_TONE).lower().strip()

    system_message = """You are a Content Quality Evaluator.
Assess on: SUCCESS CRITERIA, TONE consistency, PLATFORM FIT, QUALITY, COMPLETENESS, VIDEO/PODCAST STRUCTURE.
Be constructive but firm. Reject mediocre content with specific feedback."""

    user_message = f"""Evaluate this content creation task.

Conversation:\n{format_conversation(state['messages'])}

Success criteria: {state['success_criteria']}
Requested tone: {tone_key} — {TONE_OPTIONS.get(tone_key, 'Not specified')}

Final response to evaluate:\n{last_response}

Decide if criteria are met, tone matches, and if user input is needed."""

    if state.get("feedback_on_work"):
        user_message += f"\nPrevious feedback: {state['feedback_on_work']}\nIf same mistakes repeat, say user input is required."

    eval_result = evaluator_llm_with_output.invoke([
        SystemMessage(content=system_message),
        HumanMessage(content=user_message),
    ])

    return {
        "messages": [{"role": "assistant", "content": f"Evaluator Feedback: {eval_result.feedback}"}],
        "feedback_on_work": eval_result.feedback,
        "success_criteria_met": eval_result.success_criteria_met,
        "user_input_needed": eval_result.user_input_needed,
    }

print("Evaluator node defined.")

## Step 10: Evaluator Router

After the evaluator judges:
- Done or need user → END
- Not done → back to worker with feedback

In [ ]:
def route_based_on_evaluation(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    return "worker"

print("Evaluator router defined.")

## Step 11: Wire the Graph

Three nodes, conditional edges, compiled with a memory checkpointer.

```
START → worker → (tools ↔ worker) → evaluator → worker again OR END
```

In [ ]:
graph_builder = StateGraph(State)

# Nodes
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools_list))
graph_builder.add_node("evaluator", evaluator)

# Edges
graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})
graph_builder.add_edge(START, "worker")

# Compile
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("Graph compiled!")

## Step 12: Visualize the Graph

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Step 13: Test It!

Run the graph with a sample content creation request.
Change the `tone`, `message`, and `success_criteria` to try different content types.

In [ ]:
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

test_state = {
    "messages": [{"role": "user", "content": "Write a LinkedIn post about why every developer should learn AI agents in 2026"}],
    "success_criteria": "Must be under 3000 characters, include at least 5 hashtags, have a strong hook in the first line, and be saved to sandbox/content/",
    "tone": "hype",
    "feedback_on_work": None,
    "success_criteria_met": False,
    "user_input_needed": False,
}

result = await graph.ainvoke(test_state, config=config)

print("\n" + "="*60)
print("WORKER REPLY:")
print("="*60)
print(result["messages"][-2].content)
print("\n" + "="*60)
print("EVALUATOR FEEDBACK:")
print("="*60)
print(result["messages"][-1].content)

## Step 14: Launch Gradio UI

Full chat interface with tone dropdown. Same graph, but interactive.

In [ ]:
tone_choices = list(TONE_OPTIONS.keys())

async def process_message(message, success_criteria, tone, history):
    tid = str(uuid.uuid4())
    config = {"configurable": {"thread_id": tid}}
    state = {
        "messages": [{"role": "user", "content": message}],
        "success_criteria": success_criteria or "The content should be engaging, well-researched, and saved to a file.",
        "tone": tone or DEFAULT_TONE,
        "feedback_on_work": None,
        "success_criteria_met": False,
        "user_input_needed": False,
    }
    result = await graph.ainvoke(state, config=config)
    user = {"role": "user", "content": message}
    reply = {"role": "assistant", "content": result["messages"][-2].content}
    feedback = {"role": "assistant", "content": result["messages"][-1].content}
    return history + [user, reply, feedback]


with gr.Blocks(title="Content Creator Sidekick", theme=gr.themes.Default(primary_hue="purple")) as demo:
    gr.Markdown("## Content Creator Sidekick")
    gr.Markdown("Write viral content for any platform. Research, draft, check limits, save.")

    chatbot = gr.Chatbot(label="Sidekick", height=400, type="messages")

    with gr.Group():
        message = gr.Textbox(show_label=False, placeholder="What content do you want? e.g. 'Write a YouTube script about Python tips'", lines=2)
        success_criteria = gr.Textbox(show_label=False, placeholder="Success criteria — e.g. 'Must include 5 hashtags, under 280 chars'")
        tone = gr.Dropdown(choices=tone_choices, value=DEFAULT_TONE, label="Tone / Voice")

    go_button = gr.Button("Create!", variant="primary")

    message.submit(process_message, [message, success_criteria, tone, chatbot], [chatbot])
    go_button.click(process_message, [message, success_criteria, tone, chatbot], [chatbot])

demo.launch()

## Try These Prompts!

**Social media:**
- `Write a Twitter thread (5 tweets) about why LangGraph is better than raw API calls` — criteria: `Each tweet under 280 chars, numbered, saved to file`
- `Write a LinkedIn post about leaving my 9-5 to learn AI` — criteria: `Hook in first line, under 3000 chars, 5 hashtags`

**Video scripts:**
- `Write a YouTube script about 5 Python automation projects for beginners` — criteria: `10 minute script, include timestamps, hook in first 30 seconds, saved to file`
- `Write a TikTok script about one mind-blowing AI trick` — criteria: `Under 60 seconds, hook in first 3 seconds`

**Podcast:**
- `Write a podcast script about the future of AI agents` — criteria: `15 minute episode, include [PAUSE] cues, 3 segments, saved to file`

**Blog:**
- `Write a blog post about getting started with LangGraph` — criteria: `At least 5 sections with subheadings, intro + conclusion, saved to file`

Try different **tones** with the same prompt to see how the output changes!

## Cleanup

Run this when you're done to close the browser.

In [ ]:
await browser.close()
await playwright.stop()
print("Browser closed.")